# Newton's method

_Machine Learning — more_

**Use curvature, not just slope. Jump to the bottom of a parabola each step.**

Gradient descent only knows the slope. It takes many small, cautious steps.

     Newton's method also uses the curvature (how the slope changes).

---

This notebook is a practice scaffold for the **Newton's method** lesson. We build it up one step at a time: run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Set up a convex cost and its derivatives

Newton's method needs three things at a point: the cost, its slope (first derivative), and its curvature (second derivative). We use the simple parabola `J(t) = 3t^2 - 12t + 7`, whose minimum sits exactly at `t = 2`. Because it is a true parabola, its curvature is constant (`J'' = 6`), which is why Newton can land on the minimum in a single jump.

In [ ]:
# Convex cost J(t) = 3t^2 - 12t + 7. Minimum at t = 2.
J   = lambda t: 3*t**2 - 12*t + 7   # cost
Jp  = lambda t: 6*t - 12            # first derivative (slope)
Jpp = lambda t: 6.0                 # second derivative (curvature, constant)

t = 5.0                             # start anywhere
print("start: t=%.6f  J=%.6f  J'=%.6f" % (t, J(t), Jp(t)))

### Step 2 — Take Newton steps

Each Newton step is `t ← t - J'(t) / J''(t)`: it divides the slope by the curvature to jump straight toward where the slope would be zero. We stop early once the slope is essentially zero. For this exact parabola the first step already lands on `t = 2` — that is the payoff of using curvature instead of a fixed learning rate.

In [ ]:
for i in range(1, 5):
    t = t - Jp(t) / Jpp(t)         # Newton step: slope / curvature
    print("step %d: t=%.6f  J=%.6f  J'=%.6f" % (i, t, J(t), Jp(t)))
    if abs(Jp(t)) < 1e-9:
        break

print("minimum at t=%.6f (exact = 2.0) reached in 1 step" % t)

## Visualize the data & results

_Can curvature steps land on the minimum of a real loss?_

### Step 1 — Build a real logistic loss to minimize

We switch from a toy parabola to a real objective: fitting a single slope weight `w` that maps standardized mean tumor radius to the probability a tumor is malignant. The objective is the average logistic (negative log-likelihood) loss. This loss is convex but **not** a parabola, so its curvature changes from point to point.

In [ ]:
from sklearn.datasets import load_breast_cancer

# REAL data: 569 breast-cancer patients, feature 0 = mean tumor radius.
bc = load_breast_cancer()
x = bc.data[:, 0]
x = (x - x.mean()) / x.std()          # standardize the radius
y = (bc.target == 0).astype(float)    # 1 = malignant, 0 = benign

b = np.log(y.mean() / (1 - y.mean())) # fixed base log-odds intercept

### Step 2 — Define the loss, gradient, and Hessian

Newton needs the same three ingredients as before, now for the logistic loss: the loss value `nll`, its slope `grad`, and its curvature `hess`. Here the Hessian depends on the predicted probabilities `p(1 - p)`, so it genuinely varies as `w` moves — unlike the constant curvature of the parabola.

In [ ]:
def nll(w):                           # average logistic loss for slope w
    z = b + w*x
    return np.mean(np.log1p(np.exp(-(2*y - 1)*z)))

def grad(w):                          # slope of the loss
    p = 1/(1 + np.exp(-(b + w*x)))
    return np.mean((p - y)*x)

def hess(w):                          # curvature of the loss
    p = 1/(1 + np.exp(-(b + w*x)))
    return np.mean(p*(1 - p)*x*x)

### Step 3 — Run Newton and plot the descent

Starting from `w = 0`, each step divides the gradient by the Hessian and we record where it lands. Because the curvature is recomputed every step, Newton needs only a handful of iterations to reach the minimum near `w = 3.64`. The plot shows the loss curve with the iterates sliding down it toward the bottom.

In [ ]:
w = 0.0
iters = [w]
for _ in range(6):
    w = w - grad(w)/hess(w)           # Newton step: slope / curvature
    iters.append(w)
    if np.isclose(grad(w), 0.0):      # gradient essentially zero, stop
        break

grid = np.linspace(-1, 6, 29)
plt.plot(grid, [nll(v) for v in grid], color='#4ea1ff', label='logistic loss NLL(w)')
plt.plot(iters, [nll(v) for v in iters], 'o-', color='#c89bff',
         label='Newton iterates (start 0 to min 3.64)')
plt.xlabel('slope weight w (on mean tumor radius)')
plt.ylabel('average logistic loss (NLL)')
plt.title('Newton iterates slide down the breast-cancer logistic loss')
plt.legend()
plt.show()